# DuckWADL ADL v2.1 — 40% Target True Scored Run

This notebook is a clean competition-run rewrite of the Duck/TAAF + DuckWADL ADL allocator.

## Correct ARC score scale

The current harness reports raw game scores on a **0.0–1.0 scale**.

For the 25-game public environment:

- target mean = **0.40**
- equivalent percentage = **40%**
- target score mass = **10.0**
- perfect-game equivalent target = **10 / 25**

For an official competition rerun with `N` discovered games:

`RUN_SCORE_MASS_TARGET = 0.40 × N`

The official run is never hard-limited to ten games. Ten is only the perfect-game-equivalent target when `N = 25`.

## v2.1 corrections

1. `ELITE` is present in both action and time budget maps.
2. Every emitted triage tier is validated before execution.
3. Action budgets are enforced against **real executed environment actions**, not ADL sample count.
4. Flatline recovery budgets are based on the current real action count.
5. A no-level trajectory cannot consume the full promoted ELITE budget indefinitely.
6. Analyzer calls are not started when their available request window has collapsed below the configured reserve.
7. Submission score validation and target calculations use the observed **0–1 score scale**.
8. A score-scale guard disables target-based early stopping if a future runtime unexpectedly emits scores above 1.
9. One real environment trajectory is used per discovered game; no hidden-game best-of-two or replay is introduced.
10. Difference memory remains same-current-game only.

## Execution loop

`observe → rank current-game evidence → model plan → execute real action → measure real state difference → update ADL → triage remaining compute`

The 40% value is an optimization target, not a promised competition score.


In [ ]:
import json
import math
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = (
    os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "")
    .strip()
    .lower()
    in {"1", "true"}
)
NOTEBOOK_START_EPOCH = time.time()

# ------------------------------------------------------------------
# Correct score scale for the current ARC-AGI-3 harness.
# Raw game scores are 0.0–1.0, so 40% == 0.40.
# ------------------------------------------------------------------
PERFECT_GAME_SCORE = 1.0
TARGET_MEAN_SCORE = float(
    os.environ.get("DUCK_TAAF_TARGET_MEAN_SCORE", "0.40")
)
TARGET_MEAN_PERCENT = 100.0 * TARGET_MEAN_SCORE

REFERENCE_GAME_COUNT = int(
    os.environ.get("DUCK_TAAF_REFERENCE_GAME_COUNT", "25")
)
REFERENCE_PERFECT_GAME_EQUIVALENTS = int(
    os.environ.get("DUCK_TAAF_REFERENCE_PERFECT_GAMES", "10")
)

if not (0.0 < TARGET_MEAN_SCORE <= PERFECT_GAME_SCORE):
    raise ValueError(
        "DUCK_TAAF_TARGET_MEAN_SCORE must be in (0, 1], "
        f"got {TARGET_MEAN_SCORE!r}"
    )
if REFERENCE_GAME_COUNT <= 0:
    raise ValueError("REFERENCE_GAME_COUNT must be positive")
if REFERENCE_PERFECT_GAME_EQUIVALENTS <= 0:
    raise ValueError(
        "REFERENCE_PERFECT_GAME_EQUIVALENTS must be positive"
    )

_reference_target = (
    PERFECT_GAME_SCORE
    * REFERENCE_PERFECT_GAME_EQUIVALENTS
    / REFERENCE_GAME_COUNT
)
if abs(_reference_target - TARGET_MEAN_SCORE) > 1e-12:
    raise ValueError(
        "Reference target mismatch: "
        f"{REFERENCE_PERFECT_GAME_EQUIVALENTS}/"
        f"{REFERENCE_GAME_COUNT} perfect games implies "
        f"{_reference_target:.9f}, not {TARGET_MEAN_SCORE:.9f}"
    )

# Explicit regression guard for the run requested here.
assert abs((10.0 / 25.0) - 0.40) < 1e-12

os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = (
    "1" if TRUE_SUBMISSION else "0"
)
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = (
    "1" if TRUE_SUBMISSION else "0"
)
os.environ["ONLY_RESET_LEVELS"] = "true"

cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry
    for entry in [
        cuda_library_path,
        *os.environ.get("LIBRARY_PATH", "").split(os.pathsep),
    ]
    if entry
)

WORKING_DIR = Path(
    os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working")
)
WORKING_DIR.mkdir(parents=True, exist_ok=True)

print(
    "DuckWADL v2.1 target: "
    f"competition_rerun={TRUE_SUBMISSION} "
    f"target_mean={TARGET_MEAN_SCORE:.6f} "
    f"target_percent={TARGET_MEAN_PERCENT:.1f}% "
    f"reference_perfect_equivalents="
    f"{REFERENCE_PERFECT_GAME_EQUIVALENTS}/"
    f"{REFERENCE_GAME_COUNT}",
    flush=True,
)


## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jeroencottaar/taaf-kaggle-source-share", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Bound generation before inference modules are imported. The original run
# allowed unlimited tool steps/output, amplifying stalls under high fan-out.
_score_runtime_env = {
    'LOCAL_ANALYZER_MAX_OUTPUT': os.environ.get('TAAF_MAX_OUTPUT_TOKENS', '8192'),
    'LOCAL_ANALYZER_TOOL_STEPS': os.environ.get('TAAF_TOOL_STEPS', '8'),
    'LOCAL_ANALYZER_TEMPERATURE': os.environ.get('TAAF_TEMPERATURE', '0.6'),
    'LOCAL_ANALYZER_TOP_P': os.environ.get('TAAF_TOP_P', '0.95'),
}
os.environ.update(_score_runtime_env)
_persisted_setup_env = json.loads(SETUP_ENV_PATH.read_text())
_persisted_setup_env.update(_score_runtime_env)
SETUP_ENV_PATH.write_text(json.dumps(_persisted_setup_env, indent=2, sort_keys=True) + '\n')
print(f'taaf.kaggle: bounded analyzer controls = {_score_runtime_env}')


## 4.1 Minimal Duck Harness compatibility

Keep the bundled solver unchanged except for the missing neutral `ACTION7` reverse mapping. No prompt, score, environment, or execution method is patched.


In [ ]:
import inference.agent.action_names as action_names
import inference.framework.solver as solver_module

action_names.MODEL_TO_ENGINE_ACTION["ACTION7"] = "ACTION7"
assert action_names.to_model_action("ACTION7") == "ACTION7"
assert action_names.to_engine_action("ACTION7") == "ACTION7"

PATCH_STATUS = {
    "patch": "minimal-action7-reverse-map-v1",
    "action7_reverse_mapping": True,
    "system_prompt_changed": False,
    "solver_methods_changed": False,
    "dataset_modified": False,
}
print(f"taaf.kaggle: compatibility={PATCH_STATUS}")


## 4.2 Hard no-prior runtime contract

Declare and verify the information boundary before loading the benchmark. Only observations, actions, rewards, transitions, and ADL decisions from the **same current game in the same current run** may influence that game's future actions. Historical trajectories, solved routes, prior submissions, and cross-game state are forbidden.


In [ ]:
NO_PRIOR_CONTRACT = {
    "allowed": [
        "same_current_run_same_game_observations",
        "same_current_run_same_game_actions",
        "same_current_run_same_game_rewards",
        "same_current_run_same_game_transitions",
        "same_current_run_same_game_post_move_adl",
        "same_current_run_same_game_dual_path_decisions",
    ],
    "forbidden": [
        "historical_transcripts",
        "yesterday_transcripts",
        "routebooks",
        "replays",
        "solved_paths",
        "hidden_labels",
        "cross_run_state",
        "cross_game_state",
        "prior_submission_state",
    ],
}
assert set(NO_PRIOR_CONTRACT["allowed"]).isdisjoint(
    NO_PRIOR_CONTRACT["forbidden"]
)
print("taaf.kaggle: strict current-game/current-run no-prior contract active")


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 6. Competition-safe runtime configuration

All games enter the normal Duck/TAAF scheduler, but they no longer receive equal compute. The solver keeps a hard per-game ceiling while the ADL allocator below supplies much smaller dynamic stop budgets for weak trajectories and progressively larger budgets for promising ones.

In [ ]:
STRICT_NO_PRIOR = True

# The public Duck/TAAF harness is throughput-sensitive. Keep four lanes for the
# proven 27B runtime. If the attached model bundle overrides this through the
# environment, the requested value is still clamped to a safe positive integer.
TARGET_CONCURRENCY = max(1, int(os.environ.get("DUCK_TAAF_CONCURRENCY", "4")))

_original_game_budget = float(
    getattr(bm.solver, "max_runtime_s_per_game", 0.0) or 0.0
)
_original_concurrency = max(
    1, int(getattr(bm.solver, "concurrency", 1) or 1)
)
bm.solver.concurrency = TARGET_CONCURRENCY

# Optional public TAAF grafts remain same-current-game only. Banking and transfer
# are deliberately excluded because this allocator must not move information
# across hidden games.
try:
    from taaf_grafts.composite import install as _install_taaf_grafts
except ModuleNotFoundError:
    _install_taaf_grafts = None

_graft_flags = {
    "shortcircuit": True,
    "efficiency": True,
    "retry_guard": True,
    "recovery": True,
    "context_window": int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768")),
}
if _install_taaf_grafts is not None:
    _install_taaf_grafts(bm, _graft_flags, expected_version=1)

assert bm.solver.concurrency == TARGET_CONCURRENCY
assert "banking" not in _graft_flags
assert "transfer" not in _graft_flags

# Per-game hard ceiling. ADL triage below normally stops most weak games far
# earlier; this ceiling protects against a single promoted game monopolizing a lane.
SOURCE_GAME_CEILING_SECONDS = (
    _original_game_budget if _original_game_budget > 0 else 1500.0
)
PER_GAME_HARD_CEILING_SECONDS = min(
    1500.0,
    max(180.0, float(os.environ.get("DUCK_TAAF_GAME_HARD_CEILING_S", SOURCE_GAME_CEILING_SECONDS))),
)
bm.solver.max_runtime_s_per_game = PER_GAME_HARD_CEILING_SECONDS

print(
    "REAL RUN CONFIG: "
    f"strict_no_prior={STRICT_NO_PRIOR} "
    f"concurrency={TARGET_CONCURRENCY} "
    f"per_game_hard_ceiling_s={PER_GAME_HARD_CEILING_SECONDS:.1f} "
    "allocation=ADL_dynamic_current_game_only",
    flush=True,
)

## 7. Integrated DuckWADL ADL + DifferenceFusion policy

This cell upgrades the Duck analyzer from prompt-only dual-path selection into an explicit ADL architecture: a current-game Difference Memory schema, semantic difference signatures, weighted DifferenceFusion, prediction calibration, and mandatory post-move learning after every committed action. Candidate A/B comparison remains internal; only the selected action touches the real environment.


In [ ]:
# === DUCKWADL HARD-WIRED LIVE ADL v4 — PYTHON-ENFORCED AFTER EVERY REAL MOVE ===
import copy
import hashlib
import json
import math
import os
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional
from inference.agent.tool_agent import AnalyzerTurnResult, ToolAgent

DUCKWADL_ADL_ENABLED = True
POST_MOVE_ADL_ENABLED = True
DUAL_PATH_CANDIDATES = 2
ADL_SCHEMA = "adl.arc3.duckwadl.hardwired-live.v4.1"

DUAL_PATH_POLICY_LOG = WORKING_DIR / "dual_path_policy_events.jsonl"
POST_MOVE_ADL_LOG = WORKING_DIR / "post_move_adl_events.jsonl"
DIFFERENCE_MEMORY_LOG = WORKING_DIR / "adl_difference_memory.jsonl"
LIVE_CONTROLLER_LOG = WORKING_DIR / "adl_live_controller.jsonl"

ADL_FUSION_WEIGHTS = {
    "legality": 0.15,
    "predicted_progress": 0.18,
    "predicted_frame_change": 0.10,
    "information_gain": 0.13,
    "novelty": 0.10,
    "causal_consistency": 0.12,
    "world_model_consistency": 0.10,
    "action_efficiency": 0.07,
    "loop_avoidance": 0.05,
}
assert abs(sum(ADL_FUSION_WEIGHTS.values()) - 1.0) < 1e-9

# Only a material evidence gap can veto the model-selected discrete action.
# Mouse/coordinate actions are never rewritten by this controller.
ADL_VETO_MARGIN = float(os.environ.get("DUCKWADL_ADL_VETO_MARGIN", "0.18"))
ADL_MIN_CONTEXT_SAMPLES_FOR_VETO = int(os.environ.get("DUCKWADL_ADL_MIN_VETO_SAMPLES", "2"))
ADL_MAX_MEMORY_CONTEXT = int(os.environ.get("DUCKWADL_ADL_CONTEXT_RECORDS", "10"))


def _clip01(x: float) -> float:
    return max(0.0, min(1.0, float(x)))


def _clip11(x: float) -> float:
    return max(-1.0, min(1.0, float(x)))


def _stable_json(value: Any) -> str:
    def clean(v):
        if isinstance(v, dict):
            return {str(k): clean(val) for k, val in sorted(v.items(), key=lambda kv: str(kv[0]))
                    if str(k).lower() not in {"timestamp", "time", "elapsed", "latency", "request_id"}}
        if isinstance(v, (list, tuple)):
            return [clean(x) for x in v]
        if isinstance(v, (str, int, float, bool)) or v is None:
            return v
        return str(v)
    return json.dumps(clean(value), sort_keys=True, separators=(",", ":"), ensure_ascii=True)


def _digest(value: Any, n: int = 16) -> str:
    return hashlib.sha256(_stable_json(value).encode("utf-8", errors="replace")).hexdigest()[:n]


def _read_state_payload(state_path: Path) -> Dict[str, Any]:
    try:
        raw = Path(state_path).read_text(encoding="utf-8", errors="replace")
        value = json.loads(raw)
        return value if isinstance(value, dict) else {"value": value}
    except Exception as exc:
        return {"unreadable_state": type(exc).__name__, "path": str(state_path)}


def _recursive_number(payload: Any, keys: set[str]) -> Optional[float]:
    if isinstance(payload, dict):
        for k, v in payload.items():
            if str(k).lower() in keys and isinstance(v, (int, float)) and not isinstance(v, bool):
                return float(v)
        for v in payload.values():
            found = _recursive_number(v, keys)
            if found is not None:
                return found
    elif isinstance(payload, (list, tuple)):
        for v in payload:
            found = _recursive_number(v, keys)
            if found is not None:
                return found
    return None


def _state_signature(state_payload: Dict[str, Any]) -> str:
    return _digest(state_payload, 20)


def _normalize_action_name(value: Any) -> str:
    text = str(value or "").strip()
    if not text:
        return ""
    return text.upper()


def _extract_action_name(action_payload: Any) -> str:
    if isinstance(action_payload, str):
        return _normalize_action_name(action_payload)
    if not isinstance(action_payload, dict):
        return _normalize_action_name(action_payload)
    for key in ("action", "action_name", "name", "type"):
        if key in action_payload:
            candidate = _normalize_action_name(action_payload.get(key))
            if candidate.startswith("ACTION") or candidate in {"RESET", "MOUSE"}:
                return candidate
    # Some ARC callbacks carry a one-key action dict.
    if len(action_payload) == 1:
        only = next(iter(action_payload))
        candidate = _normalize_action_name(only)
        if candidate.startswith("ACTION") or candidate in {"RESET", "MOUSE"}:
            return candidate
    return ""


def _replace_action_name(action_payload: Any, new_name: str) -> Any:
    new_name = _normalize_action_name(new_name)
    if isinstance(action_payload, str):
        return new_name
    if not isinstance(action_payload, dict):
        return action_payload
    payload = copy.deepcopy(action_payload)
    for key in ("action", "action_name", "name", "type"):
        if key in payload:
            old = _normalize_action_name(payload.get(key))
            if old.startswith("ACTION") or old in {"RESET", "MOUSE"}:
                payload[key] = new_name
                return payload
    # Do not invent a schema when the callback shape is unknown.
    return action_payload


@dataclass
class ADLDifferenceSignature:
    step: int
    action: str
    state_signature: str = ""
    next_state_signature: str = ""
    state_changed: str = "uncertain"
    score_delta: Optional[float] = None
    level_delta: Optional[float] = None
    prediction_match: str = "uncertain"
    information_gain: float = 0.0
    progress_value: float = 0.0
    loop_signal: bool = False
    novel_transition: str = "uncertain"
    lesson: str = ""
    next_bias: str = "neutral"
    evidence_strength: float = 0.0
    model_selected_action: str = ""
    controller_executed_action: str = ""
    controller_vetoed: bool = False
    controller_score: float = 0.0

    def validate(self) -> None:
        if self.step < 0:
            raise ValueError("ADL step must be non-negative")
        self.information_gain = _clip01(self.information_gain)
        self.progress_value = _clip11(self.progress_value)
        self.evidence_strength = _clip01(self.evidence_strength)
        self.controller_score = _clip01(self.controller_score)
        if self.next_bias not in {"exploit", "explore", "neutral"}:
            raise ValueError("next_bias must be exploit/explore/neutral")


class CurrentGameDifferenceMemory:
    """Strictly current-game ADL ledger used live by the Python controller."""
    def __init__(self, game_id: str):
        self.game_id = str(game_id)
        self.records: List[ADLDifferenceSignature] = []
        self.by_context_action: Dict[str, List[ADLDifferenceSignature]] = {}
        self.by_action: Dict[str, List[ADLDifferenceSignature]] = {}
        self.visited_state_counts: Dict[str, int] = {}
        self.transition_counts: Dict[str, int] = {}
        self.prediction_hits = 0.0
        self.prediction_total = 0

    @staticmethod
    def _key(state_signature: str, action: str) -> str:
        return f"{state_signature}|{_normalize_action_name(action)}"

    def add(self, record: ADLDifferenceSignature) -> None:
        record.validate()
        self.records.append(record)
        key = self._key(record.state_signature, record.action)
        self.by_context_action.setdefault(key, []).append(record)
        self.by_action.setdefault(record.action, []).append(record)
        if record.next_state_signature:
            self.visited_state_counts[record.next_state_signature] = self.visited_state_counts.get(record.next_state_signature, 0) + 1
        transition_key = f"{record.state_signature}|{record.action}|{record.next_state_signature}"
        self.transition_counts[transition_key] = self.transition_counts.get(transition_key, 0) + 1
        if record.prediction_match in {"yes", "partial", "no"}:
            self.prediction_total += 1
            self.prediction_hits += {"yes": 1.0, "partial": 0.5, "no": 0.0}[record.prediction_match]

    def context_records(self, state_signature: str, action: str) -> List[ADLDifferenceSignature]:
        return self.by_context_action.get(self._key(state_signature, action), [])

    def action_records(self, action: str) -> List[ADLDifferenceSignature]:
        return self.by_action.get(_normalize_action_name(action), [])

    def empirical_action_value(self, action: str, state_signature: str = "") -> float:
        local = self.context_records(state_signature, action) if state_signature else []
        rows = local or self.action_records(action)
        if not rows:
            return 0.0
        weights = [max(0.05, r.evidence_strength) for r in rows]
        return sum(r.progress_value * w for r, w in zip(rows, weights)) / sum(weights)

    def repeated_noop_rate(self, state_signature: str, action: str) -> float:
        rows = self.context_records(state_signature, action)
        if not rows:
            return 0.0
        noops = sum(r.state_changed == "no" for r in rows)
        return noops / len(rows)

    def loop_rate(self, state_signature: str, action: str) -> float:
        rows = self.context_records(state_signature, action)
        if not rows:
            return 0.0
        return sum(bool(r.loop_signal) for r in rows) / len(rows)

    def prediction_accuracy(self) -> float:
        return self.prediction_hits / self.prediction_total if self.prediction_total else 0.0

    def novelty_for_action(self, state_signature: str, action: str) -> float:
        n = len(self.context_records(state_signature, action))
        return 1.0 / (1.0 + n)

    def controller_metrics(self, state_signature: str, action: str, legal: bool = True) -> Dict[str, float]:
        local = self.context_records(state_signature, action)
        global_rows = self.action_records(action)
        empirical = self.empirical_action_value(action, state_signature)
        progress01 = _clip01((empirical + 1.0) / 2.0)
        noop = self.repeated_noop_rate(state_signature, action)
        loop = self.loop_rate(state_signature, action)
        novelty = self.novelty_for_action(state_signature, action)
        sample_strength = _clip01(len(local) / 3.0)
        changed_rate = 0.5
        if local:
            known = [r for r in local if r.state_changed in {"yes", "no"}]
            if known:
                changed_rate = sum(r.state_changed == "yes" for r in known) / len(known)
        info = 0.5
        if local:
            info = sum(r.information_gain for r in local) / len(local)
        elif global_rows:
            info = sum(r.information_gain for r in global_rows[-8:]) / min(8, len(global_rows))
        causal = 0.5 + 0.5 * sample_strength if empirical > 0 else 0.5 * (1.0 - sample_strength * max(0.0, -empirical))
        return {
            "legality": 1.0 if legal else 0.0,
            "predicted_progress": progress01,
            "predicted_frame_change": _clip01(changed_rate),
            "information_gain": _clip01(max(info, novelty * 0.65)),
            "novelty": _clip01(novelty),
            "causal_consistency": _clip01(causal),
            "world_model_consistency": _clip01(0.5 + 0.5 * sample_strength if empirical >= 0 else 0.5 - 0.4 * sample_strength),
            "action_efficiency": _clip01(1.0 - noop),
            "loop_avoidance": _clip01(1.0 - loop),
        }

    def controller_score(self, state_signature: str, action: str, legal: bool = True) -> float:
        return adl_fusion_utility(self.controller_metrics(state_signature, action, legal=legal))

    def rank_actions(self, state_signature: str, valid_actions: List[str]) -> List[Dict[str, Any]]:
        rows = []
        for action in valid_actions:
            name = _normalize_action_name(action)
            metrics = self.controller_metrics(state_signature, name, legal=True)
            rows.append({
                "action": name,
                "score": adl_fusion_utility(metrics),
                "samples": len(self.context_records(state_signature, name)),
                "empirical_progress": self.empirical_action_value(name, state_signature),
                "noop_rate": self.repeated_noop_rate(state_signature, name),
                "loop_rate": self.loop_rate(state_signature, name),
                "metrics": metrics,
            })
        rows.sort(key=lambda r: (r["score"], r["empirical_progress"], -r["noop_rate"]), reverse=True)
        return rows

    def compact_context(self, state_signature: str, valid_actions: List[str], limit: int = ADL_MAX_MEMORY_CONTEXT) -> str:
        ranking = self.rank_actions(state_signature, valid_actions)
        recent = self.records[-max(1, int(limit)):]
        payload = {
            "schema": ADL_SCHEMA,
            "game_id": self.game_id,
            "state_signature": state_signature,
            "records_seen": len(self.records),
            "prediction_accuracy": round(self.prediction_accuracy(), 4),
            "controller_ranking": ranking,
            "recent_differences": [asdict(r) for r in recent],
        }
        return json.dumps(payload, sort_keys=True, separators=(",", ":"))


def adl_fusion_utility(metrics: Dict[str, float]) -> float:
    total = 0.0
    for key, weight in ADL_FUSION_WEIGHTS.items():
        value = _clip01(metrics.get(key, 0.0))
        total += weight * value
    return _clip01(total)


def _infer_transition_record(*, step: int, pre_state: Dict[str, Any], post_state: Dict[str, Any], result: Any,
                             model_action: str, executed_action: str, controller_vetoed: bool,
                             controller_score: float, memory: CurrentGameDifferenceMemory) -> ADLDifferenceSignature:
    pre_sig = _state_signature(pre_state)
    post_sig = _state_signature(post_state)
    changed = pre_sig != post_sig
    score_before = _recursive_number(pre_state, {"score", "reward", "final_score"})
    score_after = _recursive_number(post_state, {"score", "reward", "final_score"})
    if score_after is None:
        score_after = _recursive_number(result, {"score", "reward", "final_score"})
    level_before = _recursive_number(pre_state, {"level", "levels_completed", "level_index"})
    level_after = _recursive_number(post_state, {"level", "levels_completed", "level_index"})
    if level_after is None:
        level_after = _recursive_number(result, {"level", "levels_completed", "level_index"})
    score_delta = (score_after - score_before) if score_before is not None and score_after is not None else None
    level_delta = (level_after - level_before) if level_before is not None and level_after is not None else None
    transition_key = f"{pre_sig}|{executed_action}|{post_sig}"
    repeated_transition = memory.transition_counts.get(transition_key, 0)
    destination_visits = memory.visited_state_counts.get(post_sig, 0)
    loop_signal = bool((not changed) or repeated_transition >= 1 or (post_sig == pre_sig) or destination_visits >= 3)
    novelty = repeated_transition == 0 and destination_visits == 0

    progress = 0.0
    if score_delta is not None:
        progress += max(-0.7, min(0.7, score_delta))
    if level_delta is not None and level_delta > 0:
        progress += 0.8
    if not changed:
        progress -= 0.25
    if loop_signal:
        progress -= 0.20
    if changed and score_delta in {None, 0.0} and (level_delta in {None, 0.0}):
        progress += 0.08
    progress = _clip11(progress)

    info_gain = 0.0
    if novelty and changed:
        info_gain = 0.8
    elif changed:
        info_gain = 0.35
    if level_delta is not None and level_delta > 0:
        info_gain = max(info_gain, 0.6)
    if not changed:
        info_gain = 0.05

    # Python cannot know the model's semantic prediction exactly without intercepting
    # model content pre-dispatch, so prediction_match remains uncertainty-aware.
    prediction_match = "uncertain"
    evidence = 0.35
    if score_delta is not None or level_delta is not None:
        evidence += 0.20
    if changed != (pre_sig == post_sig):
        evidence += 0.10
    if repeated_transition >= 1:
        evidence += 0.15
    evidence = _clip01(evidence)

    if progress > 0.15:
        next_bias = "exploit"
    elif progress < -0.10 or info_gain > 0.55:
        next_bias = "explore"
    else:
        next_bias = "neutral"

    lesson = (
        f"{pre_sig[:8]} + {executed_action} -> {post_sig[:8]}; "
        f"changed={changed}; progress={progress:.3f}; info={info_gain:.3f}; loop={loop_signal}"
    )
    return ADLDifferenceSignature(
        step=int(step),
        action=executed_action,
        state_signature=pre_sig,
        next_state_signature=post_sig,
        state_changed="yes" if changed else "no",
        score_delta=score_delta,
        level_delta=level_delta,
        prediction_match=prediction_match,
        information_gain=info_gain,
        progress_value=progress,
        loop_signal=loop_signal,
        novel_transition="yes" if novelty else "no",
        lesson=lesson,
        next_bias=next_bias,
        evidence_strength=evidence,
        model_selected_action=model_action,
        controller_executed_action=executed_action,
        controller_vetoed=bool(controller_vetoed),
        controller_score=controller_score,
    )


DUCKWADL_ADL_CONTRACT = r"""
DUCKWADL HARD-WIRED LIVE ADL v4

Python now maintains a current-game DifferenceMemory and supplies a controller ranking before
this analyzer turn. Use that ranking as empirical evidence, not as an oracle. You still form
exactly two candidates from the SAME current observation: A=EXPLOIT and B=EXPLORE.

Before the real tool action, visibly emit DUAL_PATH_DECISION with both candidates, their
predictions, component scores and SELECT. Then issue exactly ONE real environment action.

After the tool returns, visibly emit POST_MOVE_ADL. Separately, the Python callback wrapper
will independently hash the real pre/post runtime state, compute no-op/loop/novelty/progress
signals, and append them to current-game memory before the next analyzer turn.

The Python controller may veto a discrete model-selected action ONLY when current-state
empirical evidence is sufficiently repeated and another legal discrete action exceeds it by
the configured margin. It never rewrites MOUSE/coordinate actions, never executes a second
candidate, never forks the environment, and never uses cross-game memory.

STRICT BOUNDARY: current game/current run only. No historical routes, prior submissions,
replays, hidden labels, source-code introspection, or cross-game ADL transfer.
""".strip()


class DuckWADLToolAgent(ToolAgent):
    """ToolAgent whose actual step_env callback is wrapped by live Python ADL."""
    def __init__(self, game_id: str = "unknown", **kwargs):
        super().__init__(**kwargs)
        self._adl_memory = CurrentGameDifferenceMemory(game_id)
        self._adl_game_id = str(game_id)
        self._adl_turn = 0
        if DUCKWADL_ADL_CONTRACT not in self._system_prompt:
            self._system_prompt = self._system_prompt.rstrip() + "\n\n" + DUCKWADL_ADL_CONTRACT

    def _controller_context(self, state_path: Path, valid_actions: List[str]) -> tuple[str, str]:
        payload = _read_state_payload(state_path)
        sig = _state_signature(payload)
        context = self._adl_memory.compact_context(sig, valid_actions)
        return sig, context

    def _choose_controller_override(self, *, state_signature: str, requested_action: str,
                                    valid_actions: List[str]) -> tuple[str, bool, List[Dict[str, Any]]]:
        requested = _normalize_action_name(requested_action)
        normalized_valid = [_normalize_action_name(a) for a in valid_actions if _normalize_action_name(a)]
        ranking = self._adl_memory.rank_actions(state_signature, normalized_valid)
        if not requested or requested not in normalized_valid or not ranking:
            return requested, False, ranking
        # Never rewrite coordinate-sensitive or reset semantics.
        if requested in {"MOUSE", "RESET"}:
            return requested, False, ranking
        best = ranking[0]
        requested_row = next((r for r in ranking if r["action"] == requested), None)
        if requested_row is None:
            return requested, False, ranking
        best_name = best["action"]
        if best_name in {"MOUSE", "RESET"} or best_name == requested:
            return requested, False, ranking
        # Require repeated evidence about at least one of the competing actions at this state.
        evidence_samples = max(int(best["samples"]), int(requested_row["samples"]))
        margin = float(best["score"] - requested_row["score"])
        harmful_repeat = requested_row["noop_rate"] >= 0.5 or requested_row["loop_rate"] >= 0.5
        if evidence_samples >= ADL_MIN_CONTEXT_SAMPLES_FOR_VETO and margin >= ADL_VETO_MARGIN and harmful_repeat:
            return best_name, True, ranking
        return requested, False, ranking

    def analyze(
        self,
        state_path: Path,
        action_num: int,
        valid_actions: List[str] | None = None,
        step_env: Callable[[Dict[str, Any]], Dict[str, Any]] | None = None,
        transcript_path: Path | None = None,
        analysis_step: int | None = None,
        transcript_updated: Callable[[str], None] | None = None,
        request_timeout_seconds: float | None = None,
        should_stop: Callable[[], bool] | None = None,
        **kwargs,
    ):
        valid_actions = list(valid_actions or [])

        # Do not launch a model request when TAAF has already reduced the
        # remaining request window below the reserve needed for a useful turn.
        min_request_window = float(
            globals().get(
                "MIN_ANALYZER_REQUEST_WINDOW_SECONDS",
                20.0,
            )
        )
        if (
            request_timeout_seconds is not None
            and float(request_timeout_seconds) < min_request_window
        ):
            print(
                "[PYTHON_ADL][YIELD] "
                f"game={self._adl_game_id} action={action_num} "
                f"request_timeout_s={float(request_timeout_seconds):.3f} "
                f"min_window_s={min_request_window:.1f}",
                flush=True,
            )
            return AnalyzerTurnResult(
                step_executed=False,
                retryable_failure=False,
                reasoning=(
                    "DuckWADL yielded before model inference because the "
                    "remaining analyzer request window was below the "
                    f"{min_request_window:.1f}s reserve."
                ),
                yielded_control=True,
            )
        state_sig, controller_context = self._controller_context(state_path, valid_actions)
        live_addendum = (
            "\n\n[PYTHON_ADL_CONTROLLER_CURRENT_GAME]\n"
            + controller_context
            + "\n[/PYTHON_ADL_CONTROLLER_CURRENT_GAME]\n"
            + "Use this current-game empirical ranking when estimating A/B DifferenceFusion."
        )
        original_prompt = self._system_prompt
        self._system_prompt = original_prompt.rstrip() + live_addendum

        wrapped_step = step_env
        if step_env is not None:
            def wrapped_step(action_payload):
                pre_state = _read_state_payload(state_path)
                pre_sig = _state_signature(pre_state)
                requested = _extract_action_name(action_payload)
                executed, vetoed, ranking = self._choose_controller_override(
                    state_signature=pre_sig,
                    requested_action=requested,
                    valid_actions=valid_actions,
                )
                actual_payload = _replace_action_name(action_payload, executed) if vetoed else action_payload
                actual_name = _extract_action_name(actual_payload) or requested or executed or "UNKNOWN"
                controller_score = self._adl_memory.controller_score(pre_sig, actual_name, legal=True) if actual_name else 0.0

                result = step_env(actual_payload)
                post_state = _read_state_payload(state_path)
                record = _infer_transition_record(
                    step=int(action_num),
                    pre_state=pre_state,
                    post_state=post_state,
                    result=result,
                    model_action=requested,
                    executed_action=actual_name,
                    controller_vetoed=vetoed,
                    controller_score=controller_score,
                    memory=self._adl_memory,
                )
                self._adl_memory.add(record)
                event = {
                    "schema": ADL_SCHEMA,
                    "game_id": self._adl_game_id,
                    "action_num": int(action_num),
                    "requested_action": requested,
                    "executed_action": actual_name,
                    "vetoed": vetoed,
                    "ranking": ranking,
                    "record": asdict(record),
                }
                with LIVE_CONTROLLER_LOG.open("a", encoding="utf-8") as fh:
                    fh.write(json.dumps(event, sort_keys=True) + "\n")
                with DIFFERENCE_MEMORY_LOG.open("a", encoding="utf-8") as fh:
                    fh.write(json.dumps({"game_id": self._adl_game_id, **asdict(record)}, sort_keys=True) + "\n")
                print(
                    f"[PYTHON_ADL][UPDATE] game={self._adl_game_id} step={action_num} "
                    f"requested={requested or '?'} executed={actual_name} veto={vetoed} "
                    f"changed={record.state_changed} progress={record.progress_value:.3f} "
                    f"loop={record.loop_signal} memory={len(self._adl_memory.records)}",
                    flush=True,
                )
                return result
            wrapped_step = wrapped_step

        try:
            # Preserve ToolAgent's exact analyzer contract; only the callback and prompt are wrapped.
            return super().analyze(
                state_path=state_path,
                action_num=action_num,
                valid_actions=valid_actions,
                step_env=wrapped_step,
                transcript_path=transcript_path,
                analysis_step=analysis_step,
                transcript_updated=transcript_updated,
                request_timeout_seconds=request_timeout_seconds,
                should_stop=should_stop,
                **kwargs,
            )
        finally:
            self._system_prompt = original_prompt
            self._adl_turn += 1


def _duckwadl_adl_analyzer_factory(game, index):
    model = (
        os.environ.get("INFERENCE_ANALYZER_MODEL")
        or os.environ.get("LOCAL_ANALYZER_MODEL_ID")
        or "vrfai/Qwen3.6-27B-FP8"
    )
    base_url = (
        os.environ.get("LOCAL_ANALYZER_BASE_URL")
        or os.environ.get("OPENAI_BASE_URL")
        or "http://127.0.0.1:1234/v1"
    )
    game_id = getattr(game, "game_id", None) or getattr(game, "id", None) or getattr(game, "env_name", None) or f"game-{index}"
    return DuckWADLToolAgent(
        game_id=str(game_id),
        model=model,
        timeout=bm.solver.analyzer_timeout,
        save_request_logs=bm.solver.save_request_logs,
        base_url=base_url,
        provider="vllm",
    )


bm.solver.analyzer_factory = _duckwadl_adl_analyzer_factory

print("DUCKWADL HARD-WIRED LIVE ADL v4.1 ACTIVE", flush=True)
print(f"ADL_SCHEMA={ADL_SCHEMA}", flush=True)
print("LIVE HOOK: ToolAgent.analyze step_env callback", flush=True)
print("BEFORE MOVE: Python current-state ranking injected into analyzer context", flush=True)
print("AFTER MOVE: Python hashes pre/post state and DifferenceMemory.add() runs synchronously", flush=True)
print(f"VETO POLICY: harmful repeated discrete action only; margin={ADL_VETO_MARGIN:.3f}; min_samples={ADL_MIN_CONTEXT_SAMPLES_FOR_VETO}", flush=True)
print("MOUSE/RESET REWRITE: DISABLED", flush=True)
print("ENVIRONMENT PASSES PER GAME: 1", flush=True)
print("CROSS-GAME DIFFERENCE MEMORY: DISABLED", flush=True)


## 7.1 ADL integration self-test

Validates DifferenceFusion normalization, difference-record constraints, and current-game memory behavior before any environment action is executed.


In [ ]:
# === HARD-WIRED LIVE ADL SELF-TEST: NO ENVIRONMENT ACTIONS ===
_test_memory = CurrentGameDifferenceMemory("self-test")
_pre = {"frame": [[0, 0], [0, 1]], "score": 0, "level": 1}
_post = {"frame": [[0, 0], [1, 0]], "score": 0, "level": 1}
_sig = _state_signature(_pre)
for i in range(2):
    rec = _infer_transition_record(
        step=i,
        pre_state=_pre,
        post_state=_pre,  # repeated no-op deliberately teaches a penalty
        result={"score": 0, "level": 1},
        model_action="ACTION1",
        executed_action="ACTION1",
        controller_vetoed=False,
        controller_score=0.5,
        memory=_test_memory,
    )
    _test_memory.add(rec)
rec2 = _infer_transition_record(
    step=2,
    pre_state=_pre,
    post_state=_post,
    result={"score": 0, "level": 1},
    model_action="ACTION2",
    executed_action="ACTION2",
    controller_vetoed=False,
    controller_score=0.5,
    memory=_test_memory,
)
rec2.progress_value = 0.5
rec2.evidence_strength = 0.8
_test_memory.add(rec2)
_rank = _test_memory.rank_actions(_sig, ["ACTION1", "ACTION2"])
assert len(_test_memory.records) == 3
assert _test_memory.repeated_noop_rate(_sig, "ACTION1") == 1.0
assert _test_memory.loop_rate(_sig, "ACTION1") == 1.0
assert _rank[0]["action"] == "ACTION2", _rank
assert 0.0 <= adl_fusion_utility({k: 0.5 for k in ADL_FUSION_WEIGHTS}) <= 1.0
assert _extract_action_name({"action": "ACTION3"}) == "ACTION3"
assert _extract_action_name("ACTION4") == "ACTION4"
print("HARD-WIRED ADL SELF-TEST PASSED")
print("learned ranking:", json.dumps(_rank, indent=2, sort_keys=True))
del _test_memory, _pre, _post, _sig, _rank, rec, rec2


## 7.2 ADL game triage and dynamic compute allocation

TAAF's `_HarnessGameSession.should_stop()` already owns normal runtime/action termination. This cell composes an additional current-game-only rule on top of that hook.

The allocator never restarts a game. It waits for a mandatory probe, evaluates the real ADL transition ledger, assigns the trajectory to `DROP`, `LOW`, `MEDIUM`, `HIGH`, or `ELITE`, and stops only when that tier's action/time budget is exhausted. A completed level is a hard promotion signal because ARC-AGI-3 rewards depth.

As the notebook approaches its global deadline the admission threshold rises automatically: medium/low trajectories are released first, then only elite trajectories remain. This turns worker concurrency into a practical compute allocator without violating the one-trajectory constraint.

In [ ]:
# === ADL TRIAGE + DYNAMIC COMPUTE ALLOCATOR v2.1 ===
from dataclasses import asdict, dataclass
import statistics

TRIAGE_SCHEMA = "adl.arc3.duck-taaf.triage-allocator.v2.1"
TRIAGE_ENABLED = (
    os.environ.get("DUCK_TAAF_TRIAGE", "1")
    .strip()
    .lower()
    not in {"0", "false", "off"}
)
TRIAGE_LOG = WORKING_DIR / "adl_triage_allocator.jsonl"
TRIAGE_TIERS = ("DROP", "LOW", "MEDIUM", "HIGH", "ELITE")

# Evidence probe. This is an ADL-sample requirement, not an action budget.
TRIAGE_PROBE_ACTIONS = max(
    4,
    int(os.environ.get("DUCK_TAAF_PROBE_ACTIONS", "12")),
)
TRIAGE_RECENT_WINDOW = max(
    3,
    int(os.environ.get("DUCK_TAAF_TRIAGE_RECENT", "6")),
)

# Priority thresholds.
TRIAGE_DROP_THRESHOLD = float(
    os.environ.get("DUCK_TAAF_DROP_THRESHOLD", "0.34")
)
TRIAGE_MEDIUM_THRESHOLD = float(
    os.environ.get("DUCK_TAAF_MEDIUM_THRESHOLD", "0.44")
)
TRIAGE_HIGH_THRESHOLD = float(
    os.environ.get("DUCK_TAAF_HIGH_THRESHOLD", "0.58")
)
TRIAGE_ELITE_THRESHOLD = float(
    os.environ.get("DUCK_TAAF_ELITE_THRESHOLD", "0.72")
)

if not (
    0.0
    <= TRIAGE_DROP_THRESHOLD
    <= TRIAGE_MEDIUM_THRESHOLD
    <= TRIAGE_HIGH_THRESHOLD
    <= TRIAGE_ELITE_THRESHOLD
    <= 1.0
):
    raise ValueError(
        "Triage thresholds must be monotonic in [0,1]: "
        f"{TRIAGE_DROP_THRESHOLD}, "
        f"{TRIAGE_MEDIUM_THRESHOLD}, "
        f"{TRIAGE_HIGH_THRESHOLD}, "
        f"{TRIAGE_ELITE_THRESHOLD}"
    )

# REAL ENVIRONMENT ACTION budgets.
TRIAGE_ACTION_BUDGETS = {
    "DROP": max(TRIAGE_PROBE_ACTIONS, 12),
    "LOW": max(TRIAGE_PROBE_ACTIONS + 10, 24),
    "MEDIUM": 64,
    "HIGH": 160,
    "ELITE": 360,
}

TRIAGE_TIME_BUDGETS = {
    "DROP": min(240.0, PER_GAME_HARD_CEILING_SECONDS),
    "LOW": min(420.0, PER_GAME_HARD_CEILING_SECONDS),
    "MEDIUM": min(780.0, PER_GAME_HARD_CEILING_SECONDS),
    "HIGH": min(1200.0, PER_GAME_HARD_CEILING_SECONDS),
    "ELITE": PER_GAME_HARD_CEILING_SECONDS,
}

# Prevent "interesting but never scoring" trajectories from consuming the
# entire promoted budget before their first real level completion.
TRIAGE_ZERO_LEVEL_HIGH_ACTION_CAP = max(
    64,
    int(os.environ.get("DUCK_TAAF_ZERO_LEVEL_HIGH_CAP", "128")),
)
TRIAGE_ZERO_LEVEL_ELITE_ACTION_CAP = max(
    TRIAGE_ZERO_LEVEL_HIGH_ACTION_CAP,
    int(os.environ.get("DUCK_TAAF_ZERO_LEVEL_ELITE_CAP", "160")),
)

# A promoted game must continue creating real progress evidence.
TRIAGE_FLATLINE_WINDOW = max(
    4,
    int(os.environ.get("DUCK_TAAF_FLATLINE_WINDOW", "6")),
)
TRIAGE_FLATLINE_EPSILON = float(
    os.environ.get("DUCK_TAAF_FLATLINE_EPSILON", "1e-6")
)
TRIAGE_FLATLINE_RECOVERY_ACTIONS_HIGH = max(
    4,
    int(os.environ.get("DUCK_TAAF_FLATLINE_RECOVERY_HIGH", "10")),
)
TRIAGE_FLATLINE_RECOVERY_ACTIONS_ELITE = max(
    6,
    int(os.environ.get("DUCK_TAAF_FLATLINE_RECOVERY_ELITE", "18")),
)
TRIAGE_FLATLINE_RECOVERY_SECONDS = max(
    60.0,
    float(os.environ.get("DUCK_TAAF_FLATLINE_RECOVERY_S", "180")),
)

# Never begin an analyzer request with a near-zero request window.
MIN_ANALYZER_REQUEST_WINDOW_SECONDS = max(
    5.0,
    float(os.environ.get("DUCK_TAAF_MIN_ANALYZER_WINDOW_S", "20")),
)

# Kaggle hard limit is 9h. Keep substantial output/finalization reserve.
NOTEBOOK_SOFT_STOP_SECONDS = float(
    os.environ.get(
        "DUCK_TAAF_NOTEBOOK_SOFT_STOP_S",
        str(8 * 60 * 60 + 20 * 60),
    )
)
NOTEBOOK_SOFT_STOP_EPOCH = (
    NOTEBOOK_START_EPOCH + NOTEBOOK_SOFT_STOP_SECONDS
)
TRIAGE_LATE_WINDOW_S = 60 * 60
TRIAGE_FINAL_WINDOW_S = 30 * 60
TRIAGE_CRITICAL_RESERVE_S = 15 * 60


def _validate_triage_configuration() -> None:
    expected = set(TRIAGE_TIERS)
    action_keys = set(TRIAGE_ACTION_BUDGETS)
    time_keys = set(TRIAGE_TIME_BUDGETS)

    problems = []

    if action_keys != expected:
        problems.append(
            "action tier mismatch: "
            f"expected={sorted(expected)} "
            f"actual={sorted(action_keys)}"
        )
    if time_keys != expected:
        problems.append(
            "time tier mismatch: "
            f"expected={sorted(expected)} "
            f"actual={sorted(time_keys)}"
        )

    action_values = [
        int(TRIAGE_ACTION_BUDGETS[tier])
        for tier in TRIAGE_TIERS
    ]
    time_values = [
        float(TRIAGE_TIME_BUDGETS[tier])
        for tier in TRIAGE_TIERS
    ]

    if any(value <= 0 for value in action_values):
        problems.append(
            f"non-positive action budget: {action_values}"
        )
    if any(value <= 0 for value in time_values):
        problems.append(
            f"non-positive time budget: {time_values}"
        )
    if any(
        left > right
        for left, right in zip(
            action_values,
            action_values[1:],
        )
    ):
        problems.append(
            f"action budgets not monotonic: {action_values}"
        )
    if any(
        left > right
        for left, right in zip(
            time_values,
            time_values[1:],
        )
    ):
        problems.append(
            f"time budgets not monotonic: {time_values}"
        )

    if "ELITE" not in TRIAGE_ACTION_BUDGETS:
        problems.append("ELITE missing from action budgets")
    if "ELITE" not in TRIAGE_TIME_BUDGETS:
        problems.append("ELITE missing from time budgets")

    if problems:
        raise RuntimeError(
            "Invalid ADL triage configuration:\n- "
            + "\n- ".join(problems)
        )

    print(
        "ADL TRIAGE CONFIG OK "
        + " ".join(
            f"{tier}=("
            f"{TRIAGE_ACTION_BUDGETS[tier]}a,"
            f"{TRIAGE_TIME_BUDGETS[tier]:.0f}s)"
            for tier in TRIAGE_TIERS
        ),
        flush=True,
    )


_validate_triage_configuration()


def _mean(values, default=0.0):
    values = [float(value) for value in values]
    if not values:
        return float(default)
    return sum(values) / len(values)


def _consecutive_flatline(rows) -> int:
    streak = 0
    for row in reversed(rows):
        progress = abs(
            float(getattr(row, "progress_value", 0.0) or 0.0)
        )
        level_delta = abs(
            float(getattr(row, "level_delta", 0.0) or 0.0)
        )
        score_delta = abs(
            float(getattr(row, "score_delta", 0.0) or 0.0)
        )

        if (
            progress <= TRIAGE_FLATLINE_EPSILON
            and level_delta <= TRIAGE_FLATLINE_EPSILON
            and score_delta <= TRIAGE_FLATLINE_EPSILON
        ):
            streak += 1
        else:
            break
    return streak


@dataclass
class TriageSnapshot:
    game_id: str
    samples: int
    action_count: int
    elapsed_seconds: float
    levels_completed: int
    priority: float
    tier: str
    action_budget: int
    time_budget_seconds: float
    state_change_rate: float
    recent_change_rate: float
    novelty_rate: float
    information_gain: float
    evidence_strength: float
    positive_progress_rate: float
    positive_progress_sum: float
    loop_rate: float
    noop_rate: float
    level_delta_sum: float
    score_delta_sum: float
    flatline_streak: int
    dynamic_decay_applied: bool
    zero_level_cap_applied: bool
    analyzer_window_remaining_seconds: float
    global_seconds_remaining: float
    global_phase: str
    stop_reason: str = ""


def _triage_snapshot(session) -> TriageSnapshot:
    memory = getattr(session.analyzer, "_adl_memory", None)
    rows = list(getattr(memory, "records", []) or [])
    samples = len(rows)
    recent = rows[-TRIAGE_RECENT_WINDOW:]

    changed = [
        1.0 if row.state_changed == "yes" else 0.0
        for row in rows
    ]
    recent_changed = [
        1.0 if row.state_changed == "yes" else 0.0
        for row in recent
    ]
    novel = [
        1.0 if row.novel_transition == "yes" else 0.0
        for row in rows
    ]
    info = [
        float(row.information_gain)
        for row in rows
    ]
    evidence = [
        float(row.evidence_strength)
        for row in rows
    ]
    positive_progress = [
        max(0.0, float(row.progress_value))
        for row in rows
    ]
    loops = [
        1.0 if bool(row.loop_signal) else 0.0
        for row in rows
    ]
    noops = [
        1.0 if row.state_changed == "no" else 0.0
        for row in rows
    ]

    level_delta_sum = sum(
        float(row.level_delta or 0.0)
        for row in rows
    )
    score_delta_sum = sum(
        float(row.score_delta or 0.0)
        for row in rows
    )

    state_change_rate = _mean(changed)
    recent_change_rate = _mean(recent_changed)
    novelty_rate = _mean(novel)
    information_gain = _mean(info)
    evidence_strength = _mean(evidence)
    positive_progress_rate = _mean(
        [
            1.0 if value > 0 else 0.0
            for value in positive_progress
        ]
    )
    positive_progress_sum = sum(positive_progress)
    loop_rate = _mean(loops)
    noop_rate = _mean(noops)
    flatline_streak = _consecutive_flatline(rows)

    priority = (
        0.15 * state_change_rate
        + 0.12 * recent_change_rate
        + 0.13 * novelty_rate
        + 0.13 * information_gain
        + 0.09 * evidence_strength
        + 0.13 * positive_progress_rate
        + 0.10 * (1.0 - loop_rate)
        + 0.07 * (1.0 - noop_rate)
        + 0.08 * min(1.0, positive_progress_sum)
    )

    run = getattr(session.game, "game_run", None)
    levels_completed = int(
        getattr(run, "levels_completed", 0) or 0
    )
    action_count = int(
        getattr(session, "action_count", 0) or 0
    )

    # Actual level/score progress dominates purely visual novelty.
    if levels_completed > 0 or level_delta_sum > 0:
        priority += 0.20
    if levels_completed >= 2:
        priority += 0.08
    if levels_completed >= 3:
        priority += 0.05
    if score_delta_sum > 0:
        priority += 0.08
    if positive_progress_sum >= 1.0:
        priority += 0.05
    if (
        loop_rate >= 0.60
        and noop_rate >= 0.50
        and levels_completed == 0
    ):
        priority -= 0.12

    priority = _clip01(priority)

    if (
        levels_completed >= 1
        or priority >= TRIAGE_ELITE_THRESHOLD
    ):
        tier = "ELITE"
    elif priority >= TRIAGE_HIGH_THRESHOLD:
        tier = "HIGH"
    elif priority >= TRIAGE_MEDIUM_THRESHOLD:
        tier = "MEDIUM"
    elif priority >= TRIAGE_DROP_THRESHOLD:
        tier = "LOW"
    else:
        tier = "DROP"

    if (
        tier not in TRIAGE_ACTION_BUDGETS
        or tier not in TRIAGE_TIME_BUDGETS
    ):
        raise RuntimeError(
            f"Allocator emitted unknown tier {tier!r}; "
            f"action_keys={sorted(TRIAGE_ACTION_BUDGETS)} "
            f"time_keys={sorted(TRIAGE_TIME_BUDGETS)}"
        )

    global_remaining = (
        NOTEBOOK_SOFT_STOP_EPOCH - time.time()
    )

    if global_remaining <= TRIAGE_CRITICAL_RESERVE_S:
        global_phase = "CRITICAL"
    elif global_remaining <= TRIAGE_FINAL_WINDOW_S:
        global_phase = "FINAL"
    elif global_remaining <= TRIAGE_LATE_WINDOW_S:
        global_phase = "LATE"
    else:
        global_phase = "NORMAL"

    action_budget = int(
        TRIAGE_ACTION_BUDGETS[tier]
    )
    time_budget = float(
        TRIAGE_TIME_BUDGETS[tier]
    )
    elapsed = max(
        0.0,
        time.monotonic() - session.started_at,
    )

    dynamic_decay_applied = False
    zero_level_cap_applied = False

    # No-level cap: state churn alone cannot buy an unlimited ELITE run.
    if (
        levels_completed == 0
        and level_delta_sum <= 0
        and score_delta_sum <= 0
    ):
        if tier == "HIGH":
            action_budget = min(
                action_budget,
                TRIAGE_ZERO_LEVEL_HIGH_ACTION_CAP,
            )
            zero_level_cap_applied = True
        elif tier == "ELITE":
            action_budget = min(
                action_budget,
                TRIAGE_ZERO_LEVEL_ELITE_ACTION_CAP,
            )
            zero_level_cap_applied = True

    # Flatline decay uses REAL action_count, not ADL sample count.
    if (
        samples >= TRIAGE_PROBE_ACTIONS
        and flatline_streak >= TRIAGE_FLATLINE_WINDOW
    ):
        if tier == "HIGH":
            action_budget = min(
                action_budget,
                action_count
                + TRIAGE_FLATLINE_RECOVERY_ACTIONS_HIGH,
            )
            time_budget = min(
                time_budget,
                elapsed
                + TRIAGE_FLATLINE_RECOVERY_SECONDS,
            )
            dynamic_decay_applied = True
        elif tier == "ELITE":
            action_budget = min(
                action_budget,
                action_count
                + TRIAGE_FLATLINE_RECOVERY_ACTIONS_ELITE,
            )
            time_budget = min(
                time_budget,
                elapsed
                + TRIAGE_FLATLINE_RECOVERY_SECONDS,
            )
            dynamic_decay_applied = True

    # Deadline pressure changes admission, not the priority score.
    if (
        global_phase == "LATE"
        and tier in {"DROP", "LOW", "MEDIUM"}
    ):
        action_budget = min(
            action_budget,
            max(action_count, TRIAGE_PROBE_ACTIONS),
        )
        time_budget = min(
            time_budget,
            max(
                120.0,
                elapsed
                + MIN_ANALYZER_REQUEST_WINDOW_SECONDS,
            ),
        )
    elif (
        global_phase == "FINAL"
        and tier != "ELITE"
    ):
        action_budget = min(
            action_budget,
            max(action_count, TRIAGE_PROBE_ACTIONS),
        )
        time_budget = min(
            time_budget,
            max(
                120.0,
                elapsed
                + MIN_ANALYZER_REQUEST_WINDOW_SECONDS,
            ),
        )
    elif global_phase == "CRITICAL":
        action_budget = min(
            action_budget,
            max(1, action_count),
        )
        time_budget = min(
            time_budget,
            max(1.0, elapsed),
        )

    analyzer_window_remaining = max(
        0.0,
        time_budget - elapsed,
    )

    return TriageSnapshot(
        game_id=str(
            getattr(session.game, "game_id", None)
            or getattr(run, "game_id", "unknown")
        ),
        samples=samples,
        action_count=action_count,
        elapsed_seconds=elapsed,
        levels_completed=levels_completed,
        priority=priority,
        tier=tier,
        action_budget=action_budget,
        time_budget_seconds=time_budget,
        state_change_rate=state_change_rate,
        recent_change_rate=recent_change_rate,
        novelty_rate=novelty_rate,
        information_gain=information_gain,
        evidence_strength=evidence_strength,
        positive_progress_rate=positive_progress_rate,
        positive_progress_sum=positive_progress_sum,
        loop_rate=loop_rate,
        noop_rate=noop_rate,
        level_delta_sum=level_delta_sum,
        score_delta_sum=score_delta_sum,
        flatline_streak=flatline_streak,
        dynamic_decay_applied=dynamic_decay_applied,
        zero_level_cap_applied=zero_level_cap_applied,
        analyzer_window_remaining_seconds=(
            analyzer_window_remaining
        ),
        global_seconds_remaining=max(
            0.0,
            global_remaining,
        ),
        global_phase=global_phase,
    )


def _write_triage_event(
    session,
    snapshot: TriageSnapshot,
    event: str,
    reason: str = "",
) -> None:
    payload = {
        "schema": TRIAGE_SCHEMA,
        "event": event,
        **asdict(snapshot),
        "stop_reason": (
            reason or snapshot.stop_reason
        ),
        "epoch": time.time(),
    }
    with TRIAGE_LOG.open(
        "a",
        encoding="utf-8",
    ) as handle:
        handle.write(
            json.dumps(
                payload,
                sort_keys=True,
            )
            + "\n"
        )


def _finalized_scores() -> list[float]:
    values = []
    for run in list(
        getattr(bm, "game_runs", []) or []
    ):
        try:
            value = float(
                getattr(run, "final_score", None)
                or 0.0
            )
        except (TypeError, ValueError):
            continue
        if math.isfinite(value):
            values.append(value)
    return values


def _finalized_score_mass() -> float:
    return sum(_finalized_scores())


_SCORE_SCALE_WARNING_EMITTED = False


def _score_scale_is_expected() -> bool:
    global _SCORE_SCALE_WARNING_EMITTED

    scores = _finalized_scores()
    expected = all(
        -1e-9
        <= value
        <= PERFECT_GAME_SCORE + 1e-6
        for value in scores
    )

    if (
        not expected
        and not _SCORE_SCALE_WARNING_EMITTED
    ):
        _SCORE_SCALE_WARNING_EMITTED = True
        print(
            "[SCORE_SCALE_GUARD] "
            "Observed a finalized score outside the expected "
            "0..1 range. Target-based early stopping is disabled "
            "for safety.",
            flush=True,
        )

    return expected


def _score_mass_target_reached() -> bool:
    target_mass = globals().get(
        "RUN_SCORE_MASS_TARGET"
    )
    if target_mass is None:
        return False
    if not _score_scale_is_expected():
        return False
    return (
        _finalized_score_mass()
        >= float(target_mass)
    )


def _triage_stop_reason(
    snapshot: TriageSnapshot,
) -> str:
    # Hard platform reserve always wins.
    if snapshot.global_phase == "CRITICAL":
        return "notebook_critical_reserve"

    # Evidence probe is allowed only while there is enough time for a
    # meaningful analyzer request.
    if (
        snapshot.analyzer_window_remaining_seconds
        < MIN_ANALYZER_REQUEST_WINDOW_SECONDS
    ):
        return (
            "analyzer_request_window_reserve_"
            + snapshot.tier.lower()
        )

    if snapshot.samples < TRIAGE_PROBE_ACTIONS:
        return ""

    if (
        snapshot.global_phase == "FINAL"
        and snapshot.tier != "ELITE"
    ):
        return "final_window_non_elite"

    if (
        snapshot.global_phase == "LATE"
        and snapshot.tier
        in {"DROP", "LOW", "MEDIUM"}
    ):
        return "late_window_below_high"

    # CRITICAL FIX: compare real action_count to action budget.
    if (
        snapshot.action_count
        >= snapshot.action_budget
    ):
        if snapshot.dynamic_decay_applied:
            return (
                "flatline_action_decay_"
                + snapshot.tier.lower()
            )
        if snapshot.zero_level_cap_applied:
            return (
                "zero_level_action_cap_"
                + snapshot.tier.lower()
            )
        return (
            "tier_action_budget_"
            + snapshot.tier.lower()
        )

    if (
        snapshot.elapsed_seconds
        >= snapshot.time_budget_seconds
    ):
        if snapshot.dynamic_decay_applied:
            return (
                "flatline_time_decay_"
                + snapshot.tier.lower()
            )
        return (
            "tier_time_budget_"
            + snapshot.tier.lower()
        )

    return ""


# Compose with the public harness terminal handling.
_ORIGINAL_HARNESS_SHOULD_STOP = (
    solver_module._HarnessGameSession.should_stop
)


def _adl_triage_should_stop(self) -> bool:
    if _ORIGINAL_HARNESS_SHOULD_STOP(self):
        return True
    if not TRIAGE_ENABLED:
        return False

    snapshot = _triage_snapshot(self)

    # Stop admitting further expensive work after the requested score mass
    # has already been achieved by finalized games.
    if _score_mass_target_reached():
        reason = "target_40_percent_score_mass_reached"
        snapshot.stop_reason = reason
        _write_triage_event(
            self,
            snapshot,
            "STOP",
            reason,
        )
        print(
            "[ADL_TRIAGE][STOP] "
            f"game={snapshot.game_id} "
            f"reason={reason} "
            f"finalized_score_mass="
            f"{_finalized_score_mass():.6f}",
            flush=True,
        )
        return True

    classification_key = (
        snapshot.tier,
        snapshot.global_phase,
        snapshot.action_budget,
        round(snapshot.priority, 2),
        snapshot.flatline_streak,
        snapshot.dynamic_decay_applied,
        snapshot.zero_level_cap_applied,
    )

    if (
        snapshot.samples >= TRIAGE_PROBE_ACTIONS
        and getattr(
            self,
            "_adl_triage_last_key",
            None,
        )
        != classification_key
    ):
        self._adl_triage_last_key = (
            classification_key
        )
        _write_triage_event(
            self,
            snapshot,
            "CLASSIFY",
        )
        print(
            "[ADL_TRIAGE][CLASSIFY] "
            f"game={snapshot.game_id} "
            f"samples={snapshot.samples} "
            f"real_actions={snapshot.action_count} "
            f"priority={snapshot.priority:.3f} "
            f"tier={snapshot.tier} "
            f"budget_actions={snapshot.action_budget} "
            f"budget_s={snapshot.time_budget_seconds:.0f} "
            f"levels={snapshot.levels_completed} "
            f"change={snapshot.state_change_rate:.2f} "
            f"novel={snapshot.novelty_rate:.2f} "
            f"loop={snapshot.loop_rate:.2f} "
            f"flatline={snapshot.flatline_streak} "
            f"decay={snapshot.dynamic_decay_applied} "
            f"zero_level_cap="
            f"{snapshot.zero_level_cap_applied} "
            f"window_s="
            f"{snapshot.analyzer_window_remaining_seconds:.1f} "
            f"phase={snapshot.global_phase}",
            flush=True,
        )

    reason = _triage_stop_reason(snapshot)
    if not reason:
        return False

    snapshot.stop_reason = reason
    _write_triage_event(
        self,
        snapshot,
        "STOP",
        reason,
    )

    run = getattr(
        self.game,
        "game_run",
        None,
    )
    if (
        run is not None
        and getattr(
            run,
            "solver_note",
            None,
        )
        is None
    ):
        run.solver_note = (
            f"adl_triage={snapshot.tier};"
            f"priority={snapshot.priority:.3f};"
            f"reason={reason};"
            f"samples={snapshot.samples};"
            f"actions={snapshot.action_count};"
            f"flatline={snapshot.flatline_streak}"
        )

    print(
        "[ADL_TRIAGE][STOP] "
        f"game={snapshot.game_id} "
        f"priority={snapshot.priority:.3f} "
        f"tier={snapshot.tier} "
        f"samples={snapshot.samples} "
        f"real_actions={snapshot.action_count} "
        f"action_budget={snapshot.action_budget} "
        f"elapsed={snapshot.elapsed_seconds:.1f}s "
        f"window_s="
        f"{snapshot.analyzer_window_remaining_seconds:.1f} "
        f"levels={snapshot.levels_completed} "
        f"flatline={snapshot.flatline_streak} "
        f"reason={reason}",
        flush=True,
    )
    return True


solver_module._HarnessGameSession.should_stop = (
    _adl_triage_should_stop
)

# ------------------------------------------------------------------
# Synthetic tests. No ARC environment actions are taken.
# ------------------------------------------------------------------
class _SyntheticAnalyzer:
    pass


class _SyntheticRun:
    levels_completed = 0
    game_id = "synthetic"


class _SyntheticGame:
    game_id = "synthetic"
    game_run = _SyntheticRun()


class _SyntheticSession:
    analyzer = _SyntheticAnalyzer()
    game = _SyntheticGame()
    action_count = TRIAGE_PROBE_ACTIONS
    started_at = time.monotonic() - 30


_syn_mem = CurrentGameDifferenceMemory(
    "synthetic"
)
for index in range(TRIAGE_PROBE_ACTIONS):
    _syn_mem.add(
        ADLDifferenceSignature(
            step=index,
            action="ACTION1",
            state_signature=f"s{index}",
            next_state_signature=f"s{index + 1}",
            state_changed="yes",
            information_gain=0.8,
            progress_value=0.4,
            loop_signal=False,
            novel_transition="yes",
            evidence_strength=0.8,
        )
    )

_SyntheticSession.analyzer._adl_memory = _syn_mem

_syn_snapshot = _triage_snapshot(
    _SyntheticSession()
)

assert (
    _syn_snapshot.samples
    == TRIAGE_PROBE_ACTIONS
)
assert (
    _syn_snapshot.priority
    >= TRIAGE_HIGH_THRESHOLD
), _syn_snapshot
assert _syn_snapshot.tier in {
    "HIGH",
    "ELITE",
}, _syn_snapshot
assert (
    _syn_snapshot.tier
    in TRIAGE_ACTION_BUDGETS
)
assert (
    _syn_snapshot.tier
    in TRIAGE_TIME_BUDGETS
)

# Real-action accounting regression:
_action_test = TriageSnapshot(
    **{
        **asdict(_syn_snapshot),
        "tier": "ELITE",
        "samples": TRIAGE_PROBE_ACTIONS,
        "action_count": 360,
        "action_budget": 360,
        "elapsed_seconds": 100.0,
        "time_budget_seconds": 1500.0,
        "analyzer_window_remaining_seconds": 1400.0,
        "global_phase": "NORMAL",
        "dynamic_decay_applied": False,
        "zero_level_cap_applied": False,
    }
)
assert (
    _triage_stop_reason(_action_test)
    == "tier_action_budget_elite"
)

# Samples alone must NOT exhaust a real action budget.
_sample_test = TriageSnapshot(
    **{
        **asdict(_syn_snapshot),
        "tier": "ELITE",
        "samples": 999,
        "action_count": 20,
        "action_budget": 360,
        "elapsed_seconds": 100.0,
        "time_budget_seconds": 1500.0,
        "analyzer_window_remaining_seconds": 1400.0,
        "global_phase": "NORMAL",
        "dynamic_decay_applied": False,
        "zero_level_cap_applied": False,
    }
)
assert _triage_stop_reason(_sample_test) == ""

# Analyzer request reserve regression.
_timeout_test = TriageSnapshot(
    **{
        **asdict(_syn_snapshot),
        "tier": "HIGH",
        "samples": TRIAGE_PROBE_ACTIONS,
        "action_count": 20,
        "action_budget": 160,
        "elapsed_seconds": 1190.0,
        "time_budget_seconds": 1200.0,
        "analyzer_window_remaining_seconds": 10.0,
        "global_phase": "NORMAL",
        "dynamic_decay_applied": False,
        "zero_level_cap_applied": False,
    }
)
assert (
    _triage_stop_reason(_timeout_test)
    == "analyzer_request_window_reserve_high"
)

print(
    "ADL TRIAGE SELF-TEST PASSED "
    f"priority={_syn_snapshot.priority:.3f} "
    f"tier={_syn_snapshot.tier} "
    f"action_budget={_syn_snapshot.action_budget} "
    f"time_budget="
    f"{_syn_snapshot.time_budget_seconds:.0f}s",
    flush=True,
)
print(
    "ADL REAL-ACTION BUDGET TEST PASSED",
    flush=True,
)
print(
    "ADL SAMPLE/REAL-ACTION SEPARATION TEST PASSED",
    flush=True,
)
print(
    "ADL ANALYZER WINDOW RESERVE TEST PASSED",
    flush=True,
)

del (
    _syn_mem,
    _syn_snapshot,
    _action_test,
    _sample_test,
    _timeout_test,
)

print(
    "ADL TRIAGE ALLOCATOR v2.1 ACTIVE "
    f"probe_samples={TRIAGE_PROBE_ACTIONS} "
    f"thresholds=("
    f"{TRIAGE_DROP_THRESHOLD:.2f},"
    f"{TRIAGE_MEDIUM_THRESHOLD:.2f},"
    f"{TRIAGE_HIGH_THRESHOLD:.2f},"
    f"{TRIAGE_ELITE_THRESHOLD:.2f}) "
    f"flatline_window={TRIAGE_FLATLINE_WINDOW} "
    f"min_analyzer_window_s="
    f"{MIN_ANALYZER_REQUEST_WINDOW_SECONDS:.1f} "
    f"soft_stop_s="
    f"{NOTEBOOK_SOFT_STOP_SECONDS:.0f}",
    flush=True,
)


## 8. Run every discovered game once with dynamic ADL allocation

Every game is opened exactly once. Weak trajectories finish immediately after their evidence budget says more compute is unlikely to pay; high-value trajectories remain in their existing environment and receive larger budgets. Worker lanes released by early stops are naturally reused by the TAAF scheduler for games that have not started yet.

In [ ]:
# === ONE-ENVIRONMENT TRUE/LOCAL EXECUTION WITH ADL v2.1 ===
import math
import re
from urllib.request import urlopen

EXPECTED_LOCAL_GAME_COUNT = 25


def _game_key(value):
    text = str(value or "").strip()
    match = re.match(
        r"([A-Za-z0-9]+)",
        text,
    )
    if not match:
        raise ValueError(
            f"Cannot derive game key from {value!r}"
        )
    return match.group(1)


def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=(
            arc_agi.OperationMode.COMPETITION
        ),
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=(
            arc_agi.OperationMode.COMPETITION
        ),
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )

    game_ids = [
        info.game_id
        for info in arcade.available_environments
    ]

    if not game_ids:
        raise RuntimeError(
            "Competition Arcade exposed no games."
        )
    if len(set(game_ids)) != len(game_ids):
        raise RuntimeError(
            "Competition Arcade exposed duplicate game IDs."
        )

    return [
        taaf.game_api.GameAPI(
            env_name=game_id,
            arcade_spec=spec,
        )
        for game_id in game_ids
    ]


def _offline_games():
    import arc_agi
    import taaf.game_api

    root = Path(
        "/kaggle/input/competitions/"
        "arc-prize-2026-arc-agi-3/"
        "environment_files"
    )
    if not root.is_dir():
        raise FileNotFoundError(
            f"Local environment root missing: {root}"
        )

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )

    game_ids = [
        info.game_id
        for info in arcade.available_environments
    ]

    if len(game_ids) != EXPECTED_LOCAL_GAME_COUNT:
        raise RuntimeError(
            f"Expected {EXPECTED_LOCAL_GAME_COUNT} "
            f"local games; found {len(game_ids)}"
        )

    return [
        taaf.game_api.GameAPI(
            env_name=game_id,
            arcade_spec=spec,
        )
        for game_id in game_ids
    ]


def _wait_for_gateway(
    base_url,
    timeout_s=900,
):
    deadline = (
        time.monotonic()
        + timeout_s
    )
    last_error = ""

    while time.monotonic() < deadline:
        try:
            with urlopen(
                f"{base_url}api/games",
                timeout=10,
            ) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)

        time.sleep(5)

    raise RuntimeError(
        "Competition gateway did not become ready: "
        f"{last_error}"
    )


def _run_score(run):
    value = float(
        getattr(run, "final_score", None)
        or 0.0
    )
    if not math.isfinite(value):
        raise RuntimeError(
            f"Non-finite game score for {run.game_id}: "
            f"{value!r}"
        )
    return value


def _run_levels(run):
    return int(
        getattr(run, "levels_completed", 0)
        or 0
    )


def _run_actions(run):
    return len(
        getattr(run, "history", ())
        or ()
    )


def _won(run):
    state = getattr(run, "state", "")
    state_name = str(
        getattr(state, "name", state)
    ).strip().upper()

    if (
        state_name.endswith("WIN")
        or state_name.endswith("WON")
    ):
        return True

    history = list(
        getattr(run, "history", ())
        or ()
    )
    if history:
        last = history[-1]
        candidate = getattr(
            last,
            "state",
            None,
        )
        candidate_name = str(
            getattr(
                candidate,
                "name",
                candidate,
            )
        ).strip().upper()

        if (
            candidate_name.endswith("WIN")
            or candidate_name.endswith("WON")
        ):
            return True

    return False


if TRUE_SUBMISSION:
    os.environ.setdefault(
        "ARC_API_KEY",
        "test-key-123",
    )
    os.environ.setdefault(
        "ARC_BASE_URL",
        "http://gateway:8001/",
    )
    _wait_for_gateway(
        os.environ["ARC_BASE_URL"]
    )
    game_apis = _competition_games()
    print(
        "OFFICIAL COMPETITION MODE: "
        f"{len(game_apis)} games, "
        "one live trajectory per game; "
        "ADL v2.1 enabled",
        flush=True,
    )
else:
    game_apis = _offline_games()
    print(
        "LOCAL MODE: "
        f"{len(game_apis)} games, "
        "one live trajectory per game; "
        "ADL v2.1 enabled",
        flush=True,
    )

RUN_GAME_COUNT = len(game_apis)
if RUN_GAME_COUNT < 1:
    raise RuntimeError(
        "No ARC-AGI-3 games discovered."
    )

# Correct 0..1 score-scale target.
RUN_SCORE_MASS_TARGET = (
    TARGET_MEAN_SCORE
    * RUN_GAME_COUNT
)
RUN_PERFECT_GAME_EQUIVALENT_TARGET = (
    RUN_SCORE_MASS_TARGET
    / PERFECT_GAME_SCORE
)

if (
    RUN_GAME_COUNT
    == REFERENCE_GAME_COUNT
):
    assert abs(
        RUN_SCORE_MASS_TARGET
        - 10.0
    ) < 1e-12
    assert abs(
        RUN_PERFECT_GAME_EQUIVALENT_TARGET
        - REFERENCE_PERFECT_GAME_EQUIVALENTS
    ) < 1e-12

print(
    "TRUE SCORE TARGET "
    f"games={RUN_GAME_COUNT} "
    f"mean_target="
    f"{TARGET_MEAN_SCORE:.6f} "
    f"percent_target="
    f"{TARGET_MEAN_PERCENT:.1f}% "
    f"score_mass_target="
    f"{RUN_SCORE_MASS_TARGET:.6f} "
    f"perfect_game_equivalents="
    f"{RUN_PERFECT_GAME_EQUIVALENT_TARGET:.3f}",
    flush=True,
)

# Exactly one environment trajectory per discovered game.
bm.games = game_apis
bm.n_passes = 1
bm.game_weights = None
bm.solver.concurrency = TARGET_CONCURRENCY
bm.solver.max_runtime_s_per_game = (
    PER_GAME_HARD_CEILING_SECONDS
)

run_manifest = {
    "schema": TRIAGE_SCHEMA,
    "competition_rerun": bool(
        TRUE_SUBMISSION
    ),
    "games": RUN_GAME_COUNT,
    "score_scale": "0_to_1",
    "perfect_game_score": PERFECT_GAME_SCORE,
    "target_mean_score": TARGET_MEAN_SCORE,
    "target_mean_percent": TARGET_MEAN_PERCENT,
    "target_score_mass": RUN_SCORE_MASS_TARGET,
    "perfect_game_equivalent_target": (
        RUN_PERFECT_GAME_EQUIVALENT_TARGET
    ),
    "reference_10_of_25_equals_0_40": True,
    "hard_limit_official_to_10_games": False,
    "environment_passes_per_game": 1,
    "concurrency": TARGET_CONCURRENCY,
    "per_game_hard_ceiling_seconds": (
        PER_GAME_HARD_CEILING_SECONDS
    ),
    "probe_samples": TRIAGE_PROBE_ACTIONS,
    "priority_thresholds": {
        "drop": TRIAGE_DROP_THRESHOLD,
        "medium": TRIAGE_MEDIUM_THRESHOLD,
        "high": TRIAGE_HIGH_THRESHOLD,
        "elite": TRIAGE_ELITE_THRESHOLD,
    },
    "real_action_budgets": (
        TRIAGE_ACTION_BUDGETS
    ),
    "time_budgets_seconds": (
        TRIAGE_TIME_BUDGETS
    ),
    "zero_level_high_action_cap": (
        TRIAGE_ZERO_LEVEL_HIGH_ACTION_CAP
    ),
    "zero_level_elite_action_cap": (
        TRIAGE_ZERO_LEVEL_ELITE_ACTION_CAP
    ),
    "flatline_window": (
        TRIAGE_FLATLINE_WINDOW
    ),
    "flatline_recovery_actions_high": (
        TRIAGE_FLATLINE_RECOVERY_ACTIONS_HIGH
    ),
    "flatline_recovery_actions_elite": (
        TRIAGE_FLATLINE_RECOVERY_ACTIONS_ELITE
    ),
    "minimum_analyzer_request_window_seconds": (
        MIN_ANALYZER_REQUEST_WINDOW_SECONDS
    ),
    "notebook_soft_stop_seconds": (
        NOTEBOOK_SOFT_STOP_SECONDS
    ),
    "strict_no_prior": True,
    "cross_game_solution_transfer": False,
    "second_environment_pass": False,
}

(
    WORKING_DIR
    / "adl_triage_run_manifest.json"
).write_text(
    json.dumps(
        run_manifest,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print(
    "ADL TRIAGE RUN START "
    f"games={RUN_GAME_COUNT} "
    f"concurrency={TARGET_CONCURRENCY} "
    f"probe_samples={TRIAGE_PROBE_ACTIONS} "
    f"hard_game_cap="
    f"{PER_GAME_HARD_CEILING_SECONDS:.0f}s "
    f"min_analyzer_window="
    f"{MIN_ANALYZER_REQUEST_WINDOW_SECONDS:.1f}s "
    f"notebook_soft_stop="
    f"{NOTEBOOK_SOFT_STOP_SECONDS:.0f}s",
    flush=True,
)

soft_end = datetime.fromtimestamp(
    NOTEBOOK_SOFT_STOP_EPOCH
)
run_error = None

try:
    await bm.run(
        soft_end_time=soft_end,
        runtime_environment=target,
        minimal_diagnostics=bool(
            TRUE_SUBMISSION
        ),
    )
except Exception as exc:
    run_error = exc
    print(
        "ADL TRIAGE benchmark raised "
        f"{type(exc).__name__}: {exc}",
        flush=True,
    )
finally:
    for command in json.loads(
        (
            BUNDLE_DIR
            / "teardown_commands.json"
        ).read_text()
    ):
        print(
            "Running teardown:",
            command,
            flush=True,
        )
        subprocess.run(
            command,
            shell=True,
            check=False,
            cwd=WORKING_DIR,
            env=_command_env(),
        )

if run_error is not None:
    raise run_error

runs = list(
    getattr(bm, "game_runs", [])
    or []
)
if len(runs) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Expected {RUN_GAME_COUNT} "
        f"finalized game runs; "
        f"found {len(runs)}"
    )

for run in runs:
    print(
        "ADL TRIAGE SCORE "
        f"game={_game_key(run.game_id)} "
        f"score={_run_score(run):.6f} "
        f"levels={_run_levels(run)} "
        f"actions={_run_actions(run)} "
        f"won={_won(run)} "
        f"note="
        f"{getattr(run, 'solver_note', None)!r}",
        flush=True,
    )


## 9. Write and validate `submission.parquet`


In [ ]:
# === VALIDATED COMPETITION ARTIFACT — v2.1 ===
import pandas as pd

rows = [
    {
        "row_id": f"{run.game_id}_0",
        "game_id": str(run.game_id),
        "end_of_game": bool(_won(run)),
        "score": float(_run_score(run)),
    }
    for run in bm.game_runs
]

if len(rows) != RUN_GAME_COUNT:
    raise RuntimeError(
        "Submission requires "
        f"{RUN_GAME_COUNT} rows; "
        f"found {len(rows)}"
    )

submission = pd.DataFrame(
    rows,
    columns=[
        "row_id",
        "game_id",
        "end_of_game",
        "score",
    ],
)

if submission.empty:
    raise RuntimeError(
        "submission.parquet would contain zero rows."
    )
if submission["row_id"].isna().any():
    raise RuntimeError(
        "Submission contains null row_id values."
    )
if submission["game_id"].isna().any():
    raise RuntimeError(
        "Submission contains null game_id values."
    )
if submission["score"].isna().any():
    raise RuntimeError(
        "Submission contains missing scores."
    )
if submission["row_id"].astype(str).duplicated().any():
    raise RuntimeError(
        "Submission contains duplicate row_id values."
    )
if submission["game_id"].astype(str).duplicated().any():
    raise RuntimeError(
        "Submission contains duplicate game IDs."
    )
if not submission["score"].map(math.isfinite).all():
    raise RuntimeError(
        "Submission contains non-finite scores."
    )
if (submission["score"] < -1e-9).any():
    raise RuntimeError(
        "Submission contains a negative game score."
    )

# Do not crash a future harness if its score scale changes. The current
# observed runtime is 0..1; unexpected >1 scores are surfaced loudly and
# target-based early stopping has already been guarded in the allocator.
score_scale_ok = bool(
    (
        submission["score"]
        <= PERFECT_GAME_SCORE + 1e-6
    ).all()
)
if not score_scale_ok:
    print(
        "[SCORE_SCALE_GUARD] "
        "submission contains a score above 1.0; "
        "writing the artifact but marking the scale "
        "as unexpected.",
        flush=True,
    )

SUBMISSION_PATH = (
    WORKING_DIR
    / "submission.parquet"
)
submission.to_parquet(
    SUBMISSION_PATH,
    index=False,
)

check = pd.read_parquet(
    SUBMISSION_PATH
)
expected_columns = [
    "row_id",
    "game_id",
    "end_of_game",
    "score",
]

if list(check.columns) != expected_columns:
    raise RuntimeError(
        "Invalid submission columns: "
        f"{list(check.columns)}; "
        f"expected={expected_columns}"
    )
if len(check) != RUN_GAME_COUNT:
    raise RuntimeError(
        "Written submission row count mismatch: "
        f"{len(check)} != {RUN_GAME_COUNT}"
    )
if (
    check["game_id"]
    .astype(str)
    .nunique()
    != RUN_GAME_COUNT
):
    raise RuntimeError(
        "Written submission has missing/"
        "duplicate games."
    )

score_mass = float(
    check["score"].sum()
)
mean_score = (
    score_mass
    / RUN_GAME_COUNT
)

print(
    "SUBMISSION READY "
    f"path={SUBMISSION_PATH} "
    f"rows={len(check)} "
    f"score_mass={score_mass:.6f} "
    f"mean_score={mean_score:.6f} "
    f"mean_percent="
    f"{100.0 * mean_score:.3f}% "
    f"target={TARGET_MEAN_SCORE:.6f} "
    f"target_mass="
    f"{RUN_SCORE_MASS_TARGET:.6f} "
    f"target_reached="
    f"{score_scale_ok and score_mass >= RUN_SCORE_MASS_TARGET} "
    f"score_scale_ok={score_scale_ok}",
    flush=True,
)


## 10. Final ADL run summary


In [ ]:
# === FINAL DUCKWADL ADL v2.1 SUMMARY ===
import collections

runs = list(bm.game_runs)
scores = [
    _run_score(run)
    for run in runs
]
levels = [
    _run_levels(run)
    for run in runs
]
actions = [
    _run_actions(run)
    for run in runs
]
wins = [
    _won(run)
    for run in runs
]

triage_rows = []
if TRIAGE_LOG.exists():
    for line in TRIAGE_LOG.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines():
        try:
            row = json.loads(line)
        except Exception:
            continue
        if isinstance(row, dict):
            triage_rows.append(row)

stop_rows = [
    row
    for row in triage_rows
    if row.get("event") == "STOP"
]
classify_rows = [
    row
    for row in triage_rows
    if row.get("event") == "CLASSIFY"
]

last_classification = {}
for row in classify_rows:
    last_classification[
        str(row.get("game_id"))
    ] = row

tier_counts = collections.Counter(
    str(
        row.get(
            "tier",
            "UNKNOWN",
        )
    )
    for row
    in last_classification.values()
)
stop_reason_counts = collections.Counter(
    str(
        row.get(
            "stop_reason",
            "unknown",
        )
    )
    for row in stop_rows
)

score_mass = sum(scores)
mean_score = (
    score_mass / len(scores)
    if scores
    else 0.0
)
score_scale_ok = all(
    -1e-9
    <= score
    <= PERFECT_GAME_SCORE + 1e-6
    for score in scores
)
target_reached = bool(
    score_scale_ok
    and score_mass
    >= RUN_SCORE_MASS_TARGET
)

summary = {
    "schema": TRIAGE_SCHEMA,
    "competition_rerun": bool(
        TRUE_SUBMISSION
    ),
    "games": len(runs),
    "won_games": sum(
        bool(value)
        for value in wins
    ),
    "score_scale": "0_to_1_expected",
    "score_scale_ok": score_scale_ok,
    "mean_score": mean_score,
    "mean_score_percent": (
        100.0 * mean_score
    ),
    "score_mass": score_mass,
    "target_mean_score": TARGET_MEAN_SCORE,
    "target_mean_percent": TARGET_MEAN_PERCENT,
    "target_score_mass": RUN_SCORE_MASS_TARGET,
    "target_reached": target_reached,
    "perfect_game_equivalent_score_mass": (
        score_mass / PERFECT_GAME_SCORE
    ),
    "perfect_game_equivalent_target": (
        RUN_PERFECT_GAME_EQUIVALENT_TARGET
    ),
    "positive_score_games": sum(
        score > 0
        for score in scores
    ),
    "total_levels_completed": sum(levels),
    "total_actions": sum(actions),
    "mean_actions_per_game": (
        sum(actions) / len(actions)
        if actions
        else 0.0
    ),
    "concurrency": TARGET_CONCURRENCY,
    "environment_passes_per_game": 1,
    "strict_no_prior": True,
    "cross_game_solution_transfer": False,
    "probe_samples": TRIAGE_PROBE_ACTIONS,
    "flatline_window": TRIAGE_FLATLINE_WINDOW,
    "minimum_analyzer_request_window_seconds": (
        MIN_ANALYZER_REQUEST_WINDOW_SECONDS
    ),
    "zero_level_high_action_cap": (
        TRIAGE_ZERO_LEVEL_HIGH_ACTION_CAP
    ),
    "zero_level_elite_action_cap": (
        TRIAGE_ZERO_LEVEL_ELITE_ACTION_CAP
    ),
    "tier_counts": dict(tier_counts),
    "triage_stops": len(stop_rows),
    "stop_reason_counts": dict(
        stop_reason_counts
    ),
    "submission_path": str(
        SUBMISSION_PATH
    ),
    "triage_log": str(TRIAGE_LOG),
    "difference_memory_log": str(
        DIFFERENCE_MEMORY_LOG
    ),
}

summary_path = (
    WORKING_DIR
    / "adl_triage_summary.json"
)
summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print("=" * 80)
print(
    "ARC-AGI-3 DUCKWADL — "
    "ADL v2.1 TRUE-SCORE RUN"
)
print(
    f"competition_rerun={TRUE_SUBMISSION}"
)
print(
    f"games={len(runs)} "
    f"score_mass={score_mass:.6f} "
    f"mean_score={mean_score:.6f} "
    f"mean_percent="
    f"{100.0 * mean_score:.3f}%"
)
print(
    f"target_mass="
    f"{RUN_SCORE_MASS_TARGET:.6f} "
    f"target_mean="
    f"{TARGET_MEAN_SCORE:.6f} "
    f"target_reached={target_reached}"
)
print(
    f"positive_games="
    f"{summary['positive_score_games']} "
    f"levels="
    f"{summary['total_levels_completed']} "
    f"actions="
    f"{summary['total_actions']}"
)
print(
    f"score_scale_ok={score_scale_ok}"
)
print(
    f"submission={SUBMISSION_PATH}"
)
print(
    f"summary={summary_path}"
)
print("=" * 80)


## 11. Structured DuckWADL ADL audit

Parses the visible per-move DifferenceFusion and `POST_MOVE_ADL` traces into a current-run difference dataset, measures coverage against committed actions, and reports prediction calibration, progress, information gain, loops, and novel transitions. The generated difference-memory file is an audit artifact and is **not** loaded across games.


In [ ]:
# === DUCKWADL STRUCTURED ADL EXTRACTION / AUDIT ===
# Converts visible PLAN/POST_MOVE_ADL traces from THIS run into a structured
# difference-learning dataset, while verifying that ADL covered the real actions.

import json
import re
from pathlib import Path

_TEXT_EXTS = {".log", ".txt", ".json", ".jsonl", ".md"}
_SKIP_NAMES = {
    "dual_path_adl_summary.json",
    "post_move_adl_audit.json",
    "duckwadl_adl_audit.json",
    "adl_difference_memory.jsonl",
}


def _adl_text_files(root: Path):
    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in _TEXT_EXTS:
            continue
        if path.name in _SKIP_NAMES:
            continue
        yield path


def _safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _extract_blocks(text: str, marker: str):
    # Capture marker payload until next blank boundary/major tagged event.
    pattern = re.compile(
        re.escape(marker) + r"\s*\n(?P<body>.*?)(?=\n\[(?:DUCKWADL|DIFFERENCEFUSION)\]|\n(?:DUAL_PATH_DECISION:|POST_MOVE_ADL:)|\Z)",
        re.S,
    )
    return [m.group("body") for m in pattern.finditer(text)]


def _parse_key_values(body: str):
    out = {}
    for line in body.splitlines():
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip().upper()
        value = value.strip()
        if key:
            out[key] = value
    return out


plan_records = []
post_records = []
files_scanned = []
for path in _adl_text_files(WORKING_DIR):
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    plan_bodies = _extract_blocks(text, "DUAL_PATH_DECISION:")
    post_bodies = _extract_blocks(text, "POST_MOVE_ADL:")
    if plan_bodies or post_bodies:
        files_scanned.append(str(path))
    plan_records.extend(_parse_key_values(body) for body in plan_bodies)
    post_records.extend(_parse_key_values(body) for body in post_bodies)

# De-duplicate repeated logger copies conservatively by key fields.
def _dedupe(records, keys):
    seen = set()
    unique = []
    for rec in records:
        sig = tuple(rec.get(k, "") for k in keys)
        if sig in seen:
            continue
        seen.add(sig)
        unique.append(rec)
    return unique

plan_records = _dedupe(plan_records, ("STEP", "A_ACTION", "B_ACTION", "SELECT"))
post_records = _dedupe(post_records, ("STEP", "ACTION", "DIFFERENCE_SIGNATURE", "LESSON"))

# Emit an explicit current-run ADL dataset. It is an audit artifact only; it is
# never loaded into another game by this notebook.
PARSED_ADL_LOG = WORKING_DIR / "adl_llm_post_move_parsed.jsonl"
with PARSED_ADL_LOG.open("w", encoding="utf-8") as f:
    for rec in post_records:
        row = {
            "schema": ADL_SCHEMA,
            "step": int(rec["STEP"]) if rec.get("STEP", "").isdigit() else rec.get("STEP"),
            "action": rec.get("ACTION"),
            "state_changed": rec.get("STATE_CHANGED"),
            "state_delta": rec.get("STATE_DELTA"),
            "score_delta": _safe_float(rec.get("SCORE_DELTA")),
            "level_delta": _safe_float(rec.get("LEVEL_DELTA")),
            "prediction_match": rec.get("PREDICTION_MATCH"),
            "information_gain": _safe_float(rec.get("INFORMATION_GAIN")),
            "progress_value": _safe_float(rec.get("PROGRESS_VALUE")),
            "loop_signal": rec.get("LOOP_SIGNAL"),
            "novel_transition": rec.get("NOVEL_TRANSITION"),
            "difference_signature": rec.get("DIFFERENCE_SIGNATURE"),
            "lesson": rec.get("LESSON"),
            "evidence_strength": _safe_float(rec.get("EVIDENCE_STRENGTH")),
            "next_bias": rec.get("NEXT_BIAS"),
        }
        f.write(json.dumps(row, sort_keys=True) + "\n")

runs = list(getattr(bm, "game_runs", []) or [])
total_actions = sum(len(getattr(run, "history", ()) or ()) for run in runs)

matches = [r.get("PREDICTION_MATCH", "").lower() for r in post_records]
known_predictions = [m for m in matches if m in {"yes", "partial", "no"}]
exact_prediction_accuracy = (
    sum(m == "yes" for m in known_predictions) / len(known_predictions)
    if known_predictions else None
)
soft_prediction_accuracy = (
    sum(1.0 if m == "yes" else 0.5 if m == "partial" else 0.0 for m in known_predictions)
    / len(known_predictions)
    if known_predictions else None
)
progress_values = [
    value for value in (_safe_float(r.get("PROGRESS_VALUE")) for r in post_records)
    if value is not None
]
info_values = [
    value for value in (_safe_float(r.get("INFORMATION_GAIN")) for r in post_records)
    if value is not None
]

post_move_coverage = len(post_records) / total_actions if total_actions else 0.0
plan_coverage = len(plan_records) / total_actions if total_actions else 0.0

live_controller_rows = []
if LIVE_CONTROLLER_LOG.exists():
    for line in LIVE_CONTROLLER_LOG.read_text(encoding="utf-8", errors="ignore").splitlines():
        try:
            row = json.loads(line)
        except Exception:
            continue
        if isinstance(row, dict):
            live_controller_rows.append(row)
live_memory_rows = []
if DIFFERENCE_MEMORY_LOG.exists():
    for line in DIFFERENCE_MEMORY_LOG.read_text(encoding="utf-8", errors="ignore").splitlines():
        try:
            row = json.loads(line)
        except Exception:
            continue
        if isinstance(row, dict):
            live_memory_rows.append(row)
live_veto_count = sum(bool(r.get("vetoed")) for r in live_controller_rows)
live_update_coverage = len(live_memory_rows) / total_actions if total_actions else 0.0

audit = {
    "schema": ADL_SCHEMA,
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": len(runs),
    "total_actions": total_actions,
    "live_python_adl_updates": len(live_memory_rows),
    "live_python_adl_coverage": live_update_coverage,
    "live_controller_vetoes": live_veto_count,
    "dual_path_decisions": len(plan_records),
    "post_move_adl_updates": len(post_records),
    "plan_coverage": plan_coverage,
    "post_move_coverage": post_move_coverage,
    "prediction_exact_accuracy": exact_prediction_accuracy,
    "prediction_soft_accuracy": soft_prediction_accuracy,
    "mean_progress_value": sum(progress_values) / len(progress_values) if progress_values else None,
    "mean_information_gain": sum(info_values) / len(info_values) if info_values else None,
    "loop_signals": sum(str(r.get("LOOP_SIGNAL", "")).lower() == "yes" for r in post_records),
    "novel_transitions": sum(str(r.get("NOVEL_TRANSITION", "")).lower() == "yes" for r in post_records),
    "llm_post_move_rows": len(post_records),
    "llm_parsed_path": str(PARSED_ADL_LOG),
    "live_difference_memory_path": str(DIFFERENCE_MEMORY_LOG),
    "live_controller_path": str(LIVE_CONTROLLER_LOG),
    "files_with_adl_trace": sorted(set(files_scanned)),
    "single_environment_pass": True,
    "cross_game_memory": False,
    "required_policy": "pre-action DifferenceFusion + POST_MOVE_ADL after every committed real move",
}

audit_path = WORKING_DIR / "duckwadl_adl_audit.json"
audit_path.write_text(json.dumps(audit, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("=" * 72)
print("DUCKWADL ADL AUDIT")
print(json.dumps(audit, indent=2, sort_keys=True))
print(f"live_difference_memory={DIFFERENCE_MEMORY_LOG}")
print(f"llm_parsed_memory={PARSED_ADL_LOG}")
print(f"live_controller={LIVE_CONTROLLER_LOG}")
print(f"audit={audit_path}")
print("=" * 72)


## 12. Allocator audit

This final audit joins the current-run ADL transition data with the triage decisions so the Kaggle log shows exactly which games were dropped, promoted, and allowed to consume the larger budgets.

In [ ]:
# === ALLOCATOR AUDIT v2.1 ===
triage_events = []
if TRIAGE_LOG.exists():
    for line in TRIAGE_LOG.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines():
        try:
            row = json.loads(line)
        except Exception:
            continue
        if isinstance(row, dict):
            triage_events.append(row)

per_game = {}

for event in triage_events:
    gid = str(
        event.get(
            "game_id",
            "unknown",
        )
    )
    slot = per_game.setdefault(
        gid,
        {
            "classifications": [],
            "stop": None,
        },
    )

    if event.get("event") == "CLASSIFY":
        slot["classifications"].append(
            {
                "samples": event.get("samples"),
                "real_action_count": event.get(
                    "action_count"
                ),
                "priority": event.get("priority"),
                "tier": event.get("tier"),
                "global_phase": event.get(
                    "global_phase"
                ),
                "action_budget": event.get(
                    "action_budget"
                ),
                "time_budget_seconds": event.get(
                    "time_budget_seconds"
                ),
                "analyzer_window_remaining_seconds": (
                    event.get(
                        "analyzer_window_remaining_seconds"
                    )
                ),
                "flatline_streak": event.get(
                    "flatline_streak"
                ),
                "dynamic_decay_applied": event.get(
                    "dynamic_decay_applied"
                ),
                "zero_level_cap_applied": event.get(
                    "zero_level_cap_applied"
                ),
            }
        )

    elif event.get("event") == "STOP":
        slot["stop"] = {
            "samples": event.get("samples"),
            "real_action_count": event.get(
                "action_count"
            ),
            "action_budget": event.get(
                "action_budget"
            ),
            "priority": event.get("priority"),
            "tier": event.get("tier"),
            "reason": event.get(
                "stop_reason"
            ),
            "levels_completed": event.get(
                "levels_completed"
            ),
            "analyzer_window_remaining_seconds": (
                event.get(
                    "analyzer_window_remaining_seconds"
                )
            ),
        }

run_by_key = {
    _game_key(run.game_id): run
    for run in bm.game_runs
}

for gid, slot in per_game.items():
    key = _game_key(gid)
    run = run_by_key.get(key)

    if run is not None:
        slot["final"] = {
            "score": _run_score(run),
            "levels": _run_levels(run),
            "actual_history_actions": (
                _run_actions(run)
            ),
        }

action_budget_overruns = []
for gid, slot in per_game.items():
    stop = slot.get("stop")
    if not stop:
        continue
    action_count = stop.get(
        "real_action_count"
    )
    action_budget = stop.get(
        "action_budget"
    )
    if (
        isinstance(action_count, int)
        and isinstance(action_budget, int)
        and action_count > action_budget + 1
    ):
        action_budget_overruns.append(
            {
                "game_id": gid,
                "action_count": action_count,
                "action_budget": action_budget,
                "reason": stop.get("reason"),
            }
        )

allocator_audit = {
    "schema": TRIAGE_SCHEMA,
    "competition_rerun": bool(
        TRUE_SUBMISSION
    ),
    "games_discovered": RUN_GAME_COUNT,
    "games_finalized": len(
        bm.game_runs
    ),
    "triage_events": len(
        triage_events
    ),
    "stopped_by_allocator": sum(
        1
        for event in triage_events
        if event.get("event") == "STOP"
    ),
    "action_budget_semantics": (
        "real_environment_actions"
    ),
    "probe_semantics": "adl_samples",
    "minimum_analyzer_request_window_seconds": (
        MIN_ANALYZER_REQUEST_WINDOW_SECONDS
    ),
    "action_budget_overruns": (
        action_budget_overruns
    ),
    "action_budget_overrun_count": len(
        action_budget_overruns
    ),
    "one_environment_per_game": True,
    "cross_game_memory": False,
    "target_mean_score": TARGET_MEAN_SCORE,
    "target_score_mass": RUN_SCORE_MASS_TARGET,
    "global_soft_stop_epoch": (
        NOTEBOOK_SOFT_STOP_EPOCH
    ),
    "per_game": per_game,
}

allocator_audit_path = (
    WORKING_DIR
    / "adl_triage_allocator_audit.json"
)
allocator_audit_path.write_text(
    json.dumps(
        allocator_audit,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print(
    "ALLOCATOR AUDIT READY: "
    f"{allocator_audit_path} "
    f"action_budget_overruns="
    f"{len(action_budget_overruns)}",
    flush=True,
)
